## 测试玩具和游戏数据集上的基线算法

### 练习目标（理念）

本笔记本对照第 7 周「价格预测」流水线，用**传统机器学习 / NLP 基线**（随机、均值、线性回归、词袋、NER、Word2Vec、集成、SVR、随机森林等）在商品文本上估计价格，并用统一的 `Tester` 比较误差与命中率。

### 数据链接（请先下载到本地再跑）

- 训练数据：https://drive.google.com/file/d/180ZI9OIdivkO0T-H1wki1-K514iYYK_n
- 测试数据：https://drive.google.com/file/d/1cW5doBO4jpbLQfZwygKFhy0CSSf6U1MQ

本地需准备 `train_lite.pkl` / `test_lite.pkl`（或按作者说明放置对应 pickle）。

### 和本课 Week 7 的关系

| 概念 | 本练习里你会看到 |
|------|------------------|
| 基线 vs 微调 LLM | 随机/常数/线性/BoW/Word2Vec 等先打底 |
| `Item` 与 prompt | `PREFIX` / `QUESTION` / `test_prompt()` |
| 评估协议 | `Tester`：绝对误差、RMSLE、绿色命中 |

### 怎么跑

1. 安装依赖（含 `sklearn`、`gensim`、`spacy`、`polire` 等），下载 pickle
2. 从上到下运行；空单元格是作者留下的分隔占位
3. 部分模型训练较慢（随机森林、XGB），请耐心等待


In [ ]:
# ========== 导入：环境、数值、可视化、计数 ==========

# os：路径与环境
import os
# math：对数误差等（Tester 里 RMSLE）
import math
# json：把商品 details 字符串解析成字典特征
import json
# random：随机基线定价、种子
import random
# dotenv：若后续需要密钥可从 .env 加载（保持原导入）
from dotenv import load_dotenv
# Hugging Face Hub 登录（本笔记本后续若拉模型可用；保持原导入）
from huggingface_hub import login
# matplotlib：散点图对比真值 vs 预测
import matplotlib.pyplot as plt
# numpy：数组与回归特征矩阵
import numpy as np
# pickle：读写 train/test 的 Item 列表
import pickle
# Counter：统计最常见特征名 / 品牌
from collections import Counter


In [ ]:
# ========== 再导入：传统表格机器学习 ==========

# pandas：把手工特征收成 DataFrame
import pandas as pd
# numpy：数值计算（与上格重复导入，保持原样）
import numpy as np
# 线性回归：最经典的表格基线
from sklearn.linear_model import LinearRegression
# 回归评估：MSE 与 R²
from sklearn.metrics import mean_squared_error, r2_score
# 特征缩放：标准化 / MinMax（后面 Word2Vec 特征会用到）
from sklearn.preprocessing import StandardScaler, MinMaxScaler


In [ ]:
# ========== NLP 相关导入：词袋与 Word2Vec ==========

# CountVectorizer：词袋（Bag-of-Words）稀疏特征
from sklearn.feature_extraction.text import CountVectorizer
# gensim Word2Vec：词向量
from gensim.models import Word2Vec
# simple_preprocess：把文档切成词列表供 Word2Vec
from gensim.utils import simple_preprocess


In [ ]:
# ========== 更强的表格/回归模型 ==========

# LinearSVR：线性支持向量回归
from sklearn.svm import LinearSVR
# 随机森林回归
from sklearn.ensemble import RandomForestRegressor
# 梯度提升回归（变量名里常写作 xgb，实为 sklearn GBR）
from sklearn.ensemble import GradientBoostingRegressor
# polire.IDW：反距离加权（保持原导入，即使本笔记本后文未用）
from polire import IDW


In [ ]:
# ========== 常量：终端彩色打印（Tester 每条样本上色）==========

# ANSI 转义：绿 / 黄 / 红 / 复位
GREEN = "\033[92m"
YELLOW = "\033[93m"
RED = "\033[91m"
RESET = "\033[0m"
# 把语义颜色名映射到转义码，供 print 使用
COLOR_MAP = {"red":RED, "orange": YELLOW, "green": GREEN}


In [ ]:
# Jupyter 魔法：让 matplotlib 图内嵌显示在输出区
%matplotlib inline


In [ ]:
# ========== Item：清洗商品文本并构造训练/测试 prompt ==========

# Optional：details / prompt 可为 None
from typing import Optional
# 用 Llama 分词器统计 token，控制样本长度
from transformers import AutoTokenizer
# 正则：清洗标点与空白
import re

# 与课程价格任务一致的基座分词器名（只用于 encode/decode 长度，不在此微调）
BASE_MODEL = "meta-llama/Meta-Llama-3.1-8B"

# 少于此 token 数 → 内容太短，丢弃
MIN_TOKENS = 150 # Any less than this, and we don't have enough useful content
# 截断上限；加上问答前缀后大约到 ~180 tokens
MAX_TOKENS = 160 # Truncate after this many tokens. Then after adding in prompt text, we will get to around 180 tokens

# 字符数下限 / 粗截断上限（按 token 上限 ×7 估算）
MIN_CHARS = 300
CEILING_CHARS = MAX_TOKENS * 7

class Item:
    """
    An Item is a cleaned, curated datapoint of a Product with a Price
    """
    
    # 类级分词器：所有 Item 共用，避免重复下载
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
    # 答案前缀（测试时靠它切开，去掉真实价格）
    PREFIX = "Price is $"
    # 用户问题句（须保持英文原文，模型/基线都吃同一 prompt）
    QUESTION = "How much does this cost to the nearest dollar?"
    # 从 details 里删掉的噪声片段列表
    REMOVALS = ['"Batteries Included?": "No"', '"Batteries Included?": "Yes"', '"Batteries Required?": "No"', '"Batteries Required?": "Yes"', "By Manufacturer", "Item", "Date First", "Package", ":", "Number of", "Best Sellers", "Number", "Product "]

    title: str
    price: float
    category: str
    token_count: int = 0
    details: Optional[str]
    prompt: Optional[str] = None
    include = False

    def __init__(self, data, price):
        # 标题与标价
        self.title = data['title']
        self.price = price
        # 解析描述/特征并决定是否纳入数据集
        self.parse(data)

    def scrub_details(self):
        """
        Clean up the details string by removing common text that doesn't add value
        """
        details = self.details
        # 逐个替换 REMOVALS 中的无价值子串
        for remove in self.REMOVALS:
            details = details.replace(remove, "")
        return details

    def scrub(self, stuff):
        """
        Clean up the provided text by removing unnecessary characters and whitespace
        Also remove words that are 7+ chars and contain numbers, as these are likely irrelevant product numbers
        """
        # 压缩标点与空白
        stuff = re.sub(r'[:\[\]"{}【】\s]+', ' ', stuff).strip()
        stuff = stuff.replace(" ,", ",").replace(",,,",",").replace(",,",",")
        words = stuff.split(' ')
        # 丢掉「又长又含数字」的疑似货号
        select = [word for word in words if len(word)<7 or not any(char.isdigit() for char in word)]
        return " ".join(select)
    
    def parse(self, data):
        """
        Parse this datapoint and if it fits within the allowed Token range,
        then set include to True
        """
        # 拼接 description / features / 清洗后的 details
        contents = '\n'.join(data['description'])
        if contents:
            contents += '\n'
        features = '\n'.join(data['features'])
        if features:
            contents += features + '\n'
        self.details = data['details']
        if self.details:
            contents += self.scrub_details() + '\n'
        # 够长才继续；先按字符粗截断再分词
        if len(contents) > MIN_CHARS:
            contents = contents[:CEILING_CHARS]
            text = f"{self.scrub(self.title)}\n{self.scrub(contents)}"
            tokens = self.tokenizer.encode(text, add_special_tokens=False)
            if len(tokens) > MIN_TOKENS:
                tokens = tokens[:MAX_TOKENS]
                text = self.tokenizer.decode(tokens)
                self.make_prompt(text)
                self.include = True

    def make_prompt(self, text):
        """
        Set the prompt instance variable to be a prompt appropriate for training
        """
        # 训练用：问题 + 文本 + 「Price is $xx.00」
        self.prompt = f"{self.QUESTION}\n\n{text}\n\n"
        self.prompt += f"{self.PREFIX}{str(round(self.price))}.00"
        self.token_count = len(self.tokenizer.encode(self.prompt, add_special_tokens=False))

    def test_prompt(self):
        """
        Return a prompt suitable for testing, with the actual price removed
        """
        # 测试用：截到 PREFIX，不泄露真实价格
        return self.prompt.split(self.PREFIX)[0] + self.PREFIX

    def __repr__(self):
        """
        Return a String version of this Item
        """
        return f"<{self.title} = ${self.price}>"


In [ ]:
# ========== 从本地 pickle 加载轻量训练/测试集 ==========

# 二进制读 train_lite.pkl → Item 列表
with open('./train_lite.pkl', 'rb') as file:
    train = pickle.load(file)

# 同理加载测试集
with open('./test_lite.pkl', 'rb') as file:
    test = pickle.load(file)


In [ ]:
# ========== Tester：统一评估任意 predictor(item)→价格 ==========

class Tester:

    def __init__(self, predictor, title=None, data=test, size=200):
        # predictor：可调用对象，输入 Item，输出猜测价格
        self.predictor = predictor
        self.data = data
        # 图表标题：默认用函数名美化
        self.title = title or predictor.__name__.replace("_", " ").title()
        # 默认评估前 size 条
        self.size = size
        self.guesses = []
        self.truths = []
        self.errors = []
        self.sles = []
        self.colors = []

    def color_for(self, error, truth):
        # 绝对误差或相对误差够小 → 绿；中等 → 橙；否则红
        if error<5 or error/truth < 0.1:
            return "green"
        elif error<15 or error/truth < 0.3:
            return "orange"
        else:
            return "red"
    
    def run_datapoint(self, i):
        datapoint = self.data[i]
        guess = self.predictor(datapoint)
        truth = datapoint.price
        error = abs(guess - truth)
        # 平方对数误差的一项（SLE），后面开方得 RMSLE
        log_error = math.log(truth+1) - math.log(guess+1)
        sle = log_error ** 2
        color = self.color_for(error, truth)
        title = datapoint.title if len(datapoint.title) <= 40 else datapoint.title[:40]+"..."
        self.guesses.append(guess)
        self.truths.append(truth)
        self.errors.append(error)
        self.sles.append(sle)
        self.colors.append(color)
        print(f"{COLOR_MAP[color]}{i+1}: Guess: ${guess:,.2f} Truth: ${truth:,.2f} Error: ${error:,.2f} SLE: {sle:,.2f} Item: {title}{RESET}")

    def chart(self, title):
        max_error = max(self.errors)
        plt.figure(figsize=(12, 8))
        max_val = max(max(self.truths), max(self.guesses))
        # y=x 参考线：完美预测应落在线上
        plt.plot([0, max_val], [0, max_val], color='deepskyblue', lw=2, alpha=0.6)
        plt.scatter(self.truths, self.guesses, s=3, c=self.colors)
        plt.xlabel('Ground Truth')
        plt.ylabel('Model Estimate')
        plt.xlim(0, max_val)
        plt.ylim(0, max_val)
        plt.title(title)
        plt.show()

    def report(self):
        # 平均绝对误差、RMSLE、绿色命中率
        average_error = sum(self.errors) / self.size
        rmsle = math.sqrt(sum(self.sles) / self.size)
        hits = sum(1 for color in self.colors if color=="green")
        title = f"{self.title} Error=${average_error:,.2f} RMSLE={rmsle:,.2f} Hits={hits/self.size*100:.1f}%"
        self.chart(title)

    def run(self):
        self.error = 0
        for i in range(self.size):
            self.run_datapoint(i)
        self.report()

    @classmethod
    def test(cls, function):
        # 一行启动：Tester.test(my_pricer)
        cls(function).run()


In [ ]:
# 最弱基线：完全随机报 1~59 美元（看「瞎猜」有多差）
def random_pricer(item):
    return random.randrange(1,60)


In [ ]:
# 固定随机种子，便于复现同一串「随机」猜测

random.seed(42)

# 用 Tester 跑随机定价基线
Tester.test(random_pricer)


In [ ]:
# 太有趣了！随机基线很差——下一个更简单却往往更强：训练集均价常数模型

# 收集所有训练价格
training_prices = [item.price for item in train]
# 算术平均作为「永远猜这个数」
training_average = sum(training_prices) / len(training_prices)

def constant_pricer(item):
    # 忽略 item 内容，恒返回训练集均价
    return training_average


In [ ]:
# 运行我们的常数预测器
Tester.test(constant_pricer)


In [ ]:
# 在每个 Item 上挂 features：把 details JSON 字符串解析成字典

for item in train:
    item.features = json.loads(item.details)
for item in test:
    item.features = json.loads(item.details)

# 看一个（下一格用 .keys()）


In [ ]:
# 查看第一条训练样本有哪些特征键
train[0].features.keys()


In [ ]:
# 统计训练集里各特征名出现频次，看 Top 常见字段

feature_count = Counter()
for item in train:
    for f in item.features.keys():
        feature_count[f]+=1

# 输出最常见的 40 个特征名
feature_count.most_common(40)


In [ ]:
# 从 Item Weight 字段解析重量（单位换算成磅）
# 不要太担心这一点：剧透警告，它在训练中不会有多大用处！

def get_weight(item):
    weight_str = item.features.get('Item Weight')
    if weight_str:
        parts = weight_str.split(' ')
        amount = float(parts[0])
        unit = parts[1].lower()
        if unit=="pounds":
            return amount
        elif unit=="ounces":
            return amount / 16
        elif unit=="grams":
            return amount / 453.592
        elif unit=="milligrams":
            return amount / 453592
        elif unit=="kilograms":
            return amount / 0.453592
        elif unit=="hundredths" and parts[2].lower()=="pounds":
            return amount / 100
        else:
            # 未知单位：打印原始串便于排查
            print(weight_str)
    return None


In [ ]:
# 对训练集算重量；丢掉解析失败的 None
weights = [get_weight(t) for t in train]
weights = [w for w in weights if w]


In [ ]:
# 有重量样本的平均值（后面缺省填充用）
average_weight = sum(weights)/len(weights)
average_weight


In [ ]:
# 取重量；缺失则回退到训练集平均重量
def get_weight_with_default(item):
    weight = get_weight(item)
    return weight or average_weight


In [ ]:
# 从 Best Sellers Rank 字典取各品类排名，再平均成一个数
def get_rank(item):
    rank_dict = item.features.get("Best Sellers Rank")
    if rank_dict:
        ranks = rank_dict.values()
        return sum(ranks)/len(ranks)
    return None


In [ ]:
# 收集非空排名并算平均，供缺省填充
ranks = [get_rank(t) for t in train]
ranks = [r for r in ranks if r]
average_rank = sum(ranks)/len(ranks)
average_rank


In [ ]:
# 排名缺失时用训练集平均排名
def get_rank_with_default(item):
    rank = get_rank(item)
    return rank or average_rank


In [ ]:
# 文本长度特征：用「测试用 prompt」字符数（不含真实价格后缀）
def get_text_length(item):
    return len(item.test_prompt())


In [ ]:
# 调查品牌：统计 Brand 字段频次

brands = Counter()
for t in train:
    brand = t.features.get("Brand")
    if brand:
        brands[brand]+=1

# 查看最常见的 40 个品牌

brands.most_common(40)


In [ ]:
# 手工列出的「电子类头部品牌」小写名（用于 0/1 特征）
TOP_ELECTRONICS_BRANDS = ["hp", "dell", "lenovo", "samsung", "asus", "sony", "canon", "apple", "intel"]
def is_top_electronics_brand(item):
    brand = item.features.get("Brand")
    # 品牌存在且小写落在名单里 → True
    return brand and brand.lower() in TOP_ELECTRONICS_BRANDS


In [ ]:
# 汇总四个人工特征，供表格线性回归使用
def get_features(item):
    return {
        "weight": get_weight_with_default(item),
        "rank": get_rank_with_default(item),
        "text_length": get_text_length(item),
        "is_top_electronics_brand": 1 if is_top_electronics_brand(item) else 0
    }


In [ ]:
# 查看训练项目中的功能：抽查第一条的特征向量
get_features(train[0])


In [ ]:
# 将我们的特征转换为 pandas 数据框的实用函数

def list_to_dataframe(items):
    # 每条 Item → 一行特征
    features = [get_features(item) for item in items]
    df = pd.DataFrame(features)
    # 追加目标列 price
    df['price'] = [item.price for item in items]
    return df

# 训练集 / 测试集各建一张表
train_df = list_to_dataframe(train)
test_df = list_to_dataframe(test)


In [ ]:
# 传统的线性回归！

# 固定随机性（本格主要影响可复现实验习惯）
np.random.seed(42)

# 单独的功能和目标
feature_columns = ['weight', 'rank', 'text_length', 'is_top_electronics_brand']

X_train = train_df[feature_columns]
y_train = train_df['price']
X_test = test_df[feature_columns]
y_test = test_df['price']

# 训练线性回归
model = LinearRegression()
model.fit(X_train, y_train)

# 打印各特征系数与截距，看方向与量级
for feature, coef in zip(feature_columns, model.coef_):
    print(f"{feature}: {coef}")
print(f"Intercept: {model.intercept_}")

# 预测测试集并评估
y_pred = model.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Mean Squared Error: {mse}")
print(f"R-squared Score: {r2}")


In [ ]:
# 预测新商品价格的函数：单条 Item → DataFrame → model.predict

def linear_regression_pricer(item):
    features = get_features(item)
    features_df = pd.DataFrame([features])
    return model.predict(features_df)[0]


In [ ]:
# 测试一下：用统一 Tester 看表格线性回归表现

Tester.test(linear_regression_pricer)


In [ ]:
# 对于接下来的几个型号，我们准备了文件和价格
# 请注意，我们使用文档的测试提示，否则我们将泄露答案！

# y：训练价格向量
prices = np.array([float(item.price) for item in train])
# X 文本：必须用 test_prompt()，不能含真实 Price is $xx
documents = [item.test_prompt() for item in train]


In [ ]:
# 将 CountVectorizer 用于词袋模型

np.random.seed(42)
# 最多 10000 维词表，去掉英文停用词
vectorizer = CountVectorizer(max_features=10000, stop_words='english')
# 拟合词表并变换训练文档 → 稀疏矩阵 X
X = vectorizer.fit_transform(documents)
# 在词袋特征上拟合线性回归
regressor = LinearRegression()
regressor.fit(X, prices)


In [ ]:
# 词袋 + 线性回归的单条预测；价格截断到 ≥0
def bow_lr_pricer(item):
    x = vectorizer.transform([item.test_prompt()])
    return max(regressor.predict(x)[0], 0)
# 预先算好整个 test 的预测，后面集成时直接查表
pred_lr = {}
for i in range(len(test)):
    pred_lr[test[i]] = bow_lr_pricer(test[i])


In [ ]:
# 测试一下：包装成 Tester 可调用的 lookup
def get_pred_lr(item):
    return pred_lr[item]
Tester.test(get_pred_lr)


In [ ]:
# 同一套词袋特征 X 上训练梯度提升回归（变量名 xgb）
xgb = GradientBoostingRegressor()
xgb.fit(X, prices)


In [ ]:
# 词袋 + GBR 单条预测，同样把负价截成 0
def bow_xgb_pricer(item):
    x = vectorizer.transform([item.test_prompt()])
    return max(xgb.predict(x)[0], 0)


In [ ]:
# 测试：缓存 test 预测后交给 Tester
pred_xgb = {}
for i in range(len(test)):
    pred_xgb[test[i]] = bow_xgb_pricer(test[i])
def get_pred_xgb(item):
    return pred_xgb[item]
Tester.test(get_pred_xgb)


In [ ]:
# 加载 spaCy 英文小模型，供命名实体识别（NER）抽词
import spacy 
nlp = spacy.load("en_core_web_sm")


In [ ]:
# 把文档里的实体词抽出来，去重后拼成「伪文档」供词袋用
def ner_doc(doc):
    # 换行变空格，避免 spaCy 受排版干扰
    d = nlp(doc.replace('\n',' '))
    ents = []
    for ent in d.ents:
        # 实体再按空格拆成 token
        ents.extend(ent.text.split(' '))
    return ' '.join(list(set(ents)))
def ner_docs(docs):
    ret = []
    for i,doc in enumerate(docs):
        ret.append(ner_doc(doc))
        # 每 1000 条打印进度与样例
        if i%1000 == 0:
            print(i, ret[-1])
    return ret


In [ ]:
# if 0：关闭「读缓存」分支，强制现场跑 NER（耗时）
if 0 and os.path.exists('docs.pkl'):
    docs2 = pickle.load(open('docs.pkl','rb'))
else:
    docs2 = ner_docs(documents)
    # pickle.dump(docs2, open('docs.pkl','wb'))


In [ ]:
# NER「伪文档」再做词袋 + 线性回归
np.random.seed(42)
vectorizer2 = CountVectorizer(max_features=10000, stop_words='english')
X2 = vectorizer2.fit_transform(docs2)
regressor2 = LinearRegression()
regressor2.fit(X2, prices)


In [ ]:
# NER 路径定价：先 ner_doc 再 transform；缓存 test 预测
def ner_pricer(item):
    x = vectorizer2.transform([ner_doc(item.test_prompt())])
    return max(regressor2.predict(x)[0], 0)
# 测试
pred_ner = {}
for i in range(len(test)):
    pred_ner[test[i]] = ner_pricer(test[i])
def get_pred_ner(item):
    return pred_ner[item]
Tester.test(get_pred_ner)


In [ ]:
# 令人惊叹的 word2vec 模型，在 gensim NLP 库中实现

np.random.seed(42)

# 预处理文档：切成小写词列表
processed_docs = [simple_preprocess(doc) for doc in documents]

# 训练 Word2Vec：400 维、窗口 5、保留低频词、8 线程
w2v_model = Word2Vec(sentences=processed_docs, vector_size=400, window=5, min_count=1, workers=8)


In [ ]:
# 对整个文档中的向量进行平均的这一步骤是我们方法的一个弱点

def document_vector(doc):
    doc_words = simple_preprocess(doc)
    # 只保留词表里有的词向量
    word_vectors = [w2v_model.wv[word] for word in doc_words if word in w2v_model.wv]
    # 有词则平均；否则零向量，维度与 vector_size 一致
    return np.mean(word_vectors, axis=0) if word_vectors else np.zeros(w2v_model.vector_size)

# 创建特征矩阵：每行一个文档向量
X_w2v = np.array([document_vector(doc) for doc in documents])


In [ ]:
# 在 word2vec 上运行线性回归

word2vec_lr_regressor = LinearRegression()
word2vec_lr_regressor.fit(X_w2v, prices)


In [ ]:
# Word2Vec 均值向量 → 线性回归；负价截 0
def word2vec_lr_pricer(item):
    doc = item.test_prompt()
    doc_vector = document_vector(doc)
    return max(0, word2vec_lr_regressor.predict([doc_vector])[0])


In [ ]:
# 缓存 W2V+LR 的 test 预测并评估
pred_lr_w2v = {}
for i in range(len(test)):
    pred_lr_w2v[test[i]] = word2vec_lr_pricer(test[i])
def get_pred_lr_w2v(item):
    return pred_lr_w2v[item]
Tester.test(get_pred_lr_w2v)


In [ ]:
# 专家荟萃：后面几格对多个模型预测做平均 / 选最近两两平均


In [ ]:
# 集成策略：看 BoW-LR / W2V-LR / BoW-XGB 三者两两距离，取最近一对的均值
def get_pred_best2_mean(item):
    v1 = pred_lr[item]
    v2 = pred_lr_w2v[item]
    v3 = pred_xgb[item]
    d1 = abs(v1-v2)
    d2 = abs(v2-v3)
    d3 = abs(v3-v1)
    if d1 <= min(d2,d3):
        v = (v1+v2)/2
    elif d2 <= min(d1,d3):
        v = (v2+v3)/2
    else:
        v = (v1+v3)/2
    return v
Tester.test(get_pred_best2_mean)


In [ ]:
# 更简单的集成：三模型算术平均
def get_pred_mean(item):
    v1 = pred_lr[item]
    v2 = pred_lr_w2v[item]
    v3 = pred_xgb[item]
    return (v1+v2+v3)/3
Tester.test(get_pred_mean)


In [ ]:
# 应用 MinMax 和标准缩放器（下一格实际选用 StandardScaler）


In [ ]:
# [MinMaxScaler, StandardScaler][1] → 选 StandardScaler，在 X_w2v 上 fit
scalar = [MinMaxScaler, StandardScaler][1]().fit(X_w2v)
# 变换得到缩放后的训练特征
X_w2v_scaled = scalar.transform(X_w2v)


In [ ]:
# 在标准化后的 Word2Vec 特征上再拟合线性回归
word2vec_lr_reg_scaled = LinearRegression().fit(X_w2v_scaled, prices)


In [ ]:
# 推理时：文档向量 → 同一 scalar.transform → 缩放版线性回归
def word2vec_lr_pricer_scaled(item):
    doc = item.test_prompt()
    doc_vector = document_vector(doc)
    doc_vector_scaled = scalar.transform([doc_vector])
    return max(0, word2vec_lr_reg_scaled.predict([doc_vector_scaled[0]])[0])

Tester.test(word2vec_lr_pricer_scaled)


In [ ]:
# 在 word2vec 上运行 XGB（实为 GradientBoostingRegressor）

word2vec_xgb_regressor = GradientBoostingRegressor()
word2vec_xgb_regressor.fit(X_w2v, prices)


In [ ]:
# W2V + GBR 单条预测
def word2vec_xgb_pricer(item):
    doc = item.test_prompt()
    doc_vector = document_vector(doc)
    return max(0, word2vec_xgb_regressor.predict([doc_vector])[0])


In [ ]:
# 评估 Word2Vec + 梯度提升
Tester.test(word2vec_xgb_pricer)


In [ ]:
# 支持向量机（线性 SVR）接在 Word2Vec 特征上

np.random.seed(42)
svr_regressor = LinearSVR()

svr_regressor.fit(X_w2v, prices)


In [ ]:
# SVR 定价；显式 float() 并截断负值
def svr_pricer(item):
    np.random.seed(42)
    doc = item.test_prompt()
    doc_vector = document_vector(doc)
    return max(float(svr_regressor.predict([doc_vector])[0]),0)


In [ ]:
# 评估线性 SVR
Tester.test(svr_pricer)


In [ ]:
# 以及强大的随机森林回归
mfile = 'random_forest_model.pkl'
# if 0：强制重新训练（不走读盘分支）
if 0 and os.path.exists(mfile):
    rf_model = pickle.load(open(mfile,'rb'))
else:
    # 100 棵树、固定随机种子、8 并行作业
    rf_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=8)
    rf_model.fit(X_w2v, prices)


In [ ]:
# 随机森林单条预测
def random_forest_pricer(item):
    doc = item.test_prompt()
    doc_vector = document_vector(doc)
    return max(0, rf_model.predict([doc_vector])[0])


In [ ]:
# 缓存 RF 预测并交给 Tester
pred_rf = {}
for i in range(len(test)):
    pred_rf[test[i]] = random_forest_pricer(test[i])
def get_pred_rf(item):
    return pred_rf[item]
Tester.test(get_pred_rf)
